# 12 - Silver: Fragile States Index Annual Fact Table

This notebook creates `silver.fact_fsi_annual`, one row per country-year with the Fragile States Index total score, rank, and 12 component scores.

Input:

- `bronze.fsi_raw`
- `silver.dim_country`
- `silver.dim_bloc_membership`

Official FSI Excel downloads currently cover 2006-2023. The silver table preserves that real source coverage and does not fabricate 1990-2005 or 2024 rows.

In [ ]:
# Cell 1 - Configuration
from datetime import datetime, timezone

from pyspark.sql import functions as F

spark.sql("USE CATALOG cemac_ecowas_aes_trade")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

loaded_at = datetime.now(timezone.utc)

INDICATOR_COLUMNS = {
    "TOTAL": "fsi_total_score",
    "C1": "security_apparatus_score",
    "C2": "factionalized_elites_score",
    "C3": "group_grievance_score",
    "E1": "economic_decline_score",
    "E2": "uneven_economic_development_score",
    "E3": "human_flight_brain_drain_score",
    "P1": "state_legitimacy_score",
    "P2": "public_services_score",
    "P3": "human_rights_rule_of_law_score",
    "S1": "demographic_pressures_score",
    "S2": "refugees_idps_score",
    "X1": "external_intervention_score",
}

COHESION_CODES = ["C1", "C2", "C3"]
ECONOMIC_CODES = ["E1", "E2", "E3"]
POLITICAL_CODES = ["P1", "P2", "P3"]
SOCIAL_CROSS_CUTTING_CODES = ["S1", "S2", "X1"]

print("Target table: silver.fact_fsi_annual")

In [ ]:
# Cell 2 - Input table checks
def table_exists(schema_name, table_name):
    return spark.sql(f"SHOW TABLES IN {schema_name} LIKE '{table_name}'").count() > 0


required_tables = [
    ("bronze", "fsi_raw"),
    ("silver", "dim_country"),
    ("silver", "dim_bloc_membership"),
]

missing_tables = [f"{schema}.{table}" for schema, table in required_tables if not table_exists(schema, table)]
if missing_tables:
    raise ValueError(f"Missing required input table(s): {missing_tables}")

print("All required tables found.")

print("FSI bronze input coverage:")
spark.sql("""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT country_iso3) AS countries,
        COUNT(DISTINCT indicator_code) AS indicators,
        MIN(year) AS earliest_year,
        MAX(year) AS latest_year,
        COUNT(value) AS non_null_values
    FROM bronze.fsi_raw
""").show()

print("FSI indicators:")
spark.sql("""
    SELECT
        indicator_code,
        indicator_name,
        category,
        COUNT(*) AS rows,
        COUNT(DISTINCT country_iso3) AS countries,
        MIN(year) AS earliest_year,
        MAX(year) AS latest_year
    FROM bronze.fsi_raw
    GROUP BY indicator_code, indicator_name, category
    ORDER BY indicator_code
""").show(truncate=False)

In [ ]:
# Cell 3 - Pivot FSI long indicators into annual country rows
fsi_raw = (
    spark.table("bronze.fsi_raw")
    .where(F.col("indicator_code").isin(list(INDICATOR_COLUMNS.keys())))
    .select(
        F.upper(F.trim(F.col("country_iso3"))).alias("country_iso3"),
        F.col("country_name_source"),
        F.col("year").cast("int").alias("year"),
        F.col("rank").cast("int").alias("rank"),
        F.upper(F.trim(F.col("indicator_code"))).alias("indicator_code"),
        F.col("indicator_name"),
        F.col("category"),
        F.col("value").cast("double").alias("value"),
        F.col("source_url"),
    )
)

pivot_exprs = [
    F.max(F.when(F.col("indicator_code") == indicator_code, F.col("value"))).alias(column_name)
    for indicator_code, column_name in INDICATOR_COLUMNS.items()
]

rank_df = (
    fsi_raw.where(F.col("indicator_code") == "TOTAL")
    .groupBy("country_iso3", "year")
    .agg(
        F.max("rank").alias("fsi_rank"),
        F.max("source_url").alias("source_url"),
        F.max("country_name_source").alias("country_name_source"),
    )
)

fsi_wide_df = (
    fsi_raw.groupBy("country_iso3", "year")
    .agg(*pivot_exprs)
    .join(rank_df, ["country_iso3", "year"], "left")
)

print(f"FSI wide rows: {fsi_wide_df.count():,}")
fsi_wide_df.show(10, truncate=False)

In [ ]:
# Cell 4 - Join dimensions and derive category scores
country_dim = spark.table("silver.dim_country").select(
    "country_key",
    "country_iso3",
    "country_name",
    F.col("region").alias("project_region"),
)

bloc_df = spark.table("silver.dim_bloc_membership").where("is_primary_analytical_bloc = true").select(
    F.col("country_iso3").alias("bloc_country_iso3"),
    F.col("bloc_code").alias("analytical_bloc_code"),
    F.col("bloc_name").alias("analytical_bloc_name"),
    F.col("valid_from").alias("bloc_valid_from"),
    F.col("valid_to").alias("bloc_valid_to"),
)

fact_df = (
    fsi_wide_df.alias("f")
    .join(country_dim.alias("c"), F.col("f.country_iso3") == F.col("c.country_iso3"), "left")
    .withColumn("year_end_date", F.expr("make_date(year, 12, 31)"))
    .join(
        bloc_df.alias("b"),
        (F.col("f.country_iso3") == F.col("b.bloc_country_iso3"))
        & (F.col("year_end_date") >= F.col("b.bloc_valid_from"))
        & (F.col("b.bloc_valid_to").isNull() | (F.col("year_end_date") <= F.col("b.bloc_valid_to"))),
        "left",
    )
    .select(
        F.col("c.country_key"),
        F.col("f.country_iso3"),
        F.coalesce(F.col("c.country_name"), F.col("f.country_name_source"), F.col("f.country_iso3")).alias("country_name"),
        F.col("f.country_name_source"),
        F.col("c.project_region"),
        F.col("b.analytical_bloc_code"),
        F.col("b.analytical_bloc_name"),
        F.col("year"),
        F.col("year_end_date"),
        F.col("fsi_rank"),
        *[F.col(column_name) for column_name in INDICATOR_COLUMNS.values()],
        F.col("source_url"),
    )
)

fact_df = (
    fact_df
    .withColumn("cohesion_score", F.col("security_apparatus_score") + F.col("factionalized_elites_score") + F.col("group_grievance_score"))
    .withColumn("economic_score", F.col("economic_decline_score") + F.col("uneven_economic_development_score") + F.col("human_flight_brain_drain_score"))
    .withColumn("political_score", F.col("state_legitimacy_score") + F.col("public_services_score") + F.col("human_rights_rule_of_law_score"))
    .withColumn("social_cross_cutting_score", F.col("demographic_pressures_score") + F.col("refugees_idps_score") + F.col("external_intervention_score"))
    .withColumn(
        "computed_total_score",
        F.col("cohesion_score") + F.col("economic_score") + F.col("political_score") + F.col("social_cross_cutting_score"),
    )
    .withColumn("total_score_difference", F.round(F.col("computed_total_score") - F.col("fsi_total_score"), 3))
    .withColumn(
        "fragility_band",
        F.when(F.col("fsi_total_score") >= 100, F.lit("very_high"))
        .when(F.col("fsi_total_score") >= 90, F.lit("high"))
        .when(F.col("fsi_total_score") >= 70, F.lit("elevated"))
        .when(F.col("fsi_total_score") >= 50, F.lit("moderate"))
        .otherwise(F.lit("low")),
    )
    .withColumn("source", F.lit("fragile_states_index"))
    .withColumn("loaded_at", F.lit(loaded_at))
)

missing_country_keys = fact_df.where(F.col("country_key").isNull()).select("country_iso3").distinct().count()
if missing_country_keys:
    fact_df.where(F.col("country_key").isNull()).select("country_iso3").distinct().show(truncate=False)
    raise ValueError(f"FSI rows missing country dimension matches: {missing_country_keys}")

print(f"Fact rows: {fact_df.count():,}")
fact_df.printSchema()
fact_df.show(10, truncate=False)

In [ ]:
# Cell 5 - Write silver.fact_fsi_annual
spark.sql("DROP TABLE IF EXISTS silver.fact_fsi_annual")

(fact_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.fact_fsi_annual"))

print("Write complete: silver.fact_fsi_annual")

In [ ]:
# Cell 6 - Verification and sanity checks
print("Overall coverage:")
spark.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT country_iso3) AS countries,
        MIN(year) AS earliest_year,
        MAX(year) AS latest_year,
        COUNT(fsi_total_score) AS non_null_total_scores,
        ROUND(MAX(ABS(total_score_difference)), 3) AS max_component_total_difference
    FROM silver.fact_fsi_annual
""").show()

print("Coverage by bloc:")
spark.sql("""
    SELECT
        analytical_bloc_code,
        COUNT(*) AS rows,
        COUNT(DISTINCT country_iso3) AS countries,
        MIN(year) AS earliest_year,
        MAX(year) AS latest_year,
        ROUND(AVG(fsi_total_score), 1) AS avg_total_score
    FROM silver.fact_fsi_annual
    GROUP BY analytical_bloc_code
    ORDER BY analytical_bloc_code
""").show(truncate=False)

print("Latest total fragility score:")
spark.sql("""
    SELECT
        analytical_bloc_code,
        country_iso3,
        country_name,
        year,
        fsi_rank,
        ROUND(fsi_total_score, 1) AS total_score,
        fragility_band,
        ROUND(cohesion_score, 1) AS cohesion,
        ROUND(economic_score, 1) AS economic,
        ROUND(political_score, 1) AS political,
        ROUND(social_cross_cutting_score, 1) AS social_cross_cutting
    FROM silver.fact_fsi_annual
    WHERE year = (SELECT MAX(year) FROM silver.fact_fsi_annual)
    ORDER BY fsi_total_score DESC
""").show(25, truncate=False)

print("Cameroon FSI trajectory:")
spark.sql("""
    SELECT
        year,
        fsi_rank,
        ROUND(fsi_total_score, 1) AS total_score,
        fragility_band,
        ROUND(cohesion_score, 1) AS cohesion,
        ROUND(political_score, 1) AS political,
        ROUND(security_apparatus_score, 1) AS security_apparatus,
        ROUND(group_grievance_score, 1) AS group_grievance
    FROM silver.fact_fsi_annual
    WHERE country_iso3 = 'CMR'
    ORDER BY year
""").show(30, truncate=False)